# Player Animation

In [1]:
from pathlib import Path
import sys
print(sys.version)
print(sys.executable)

# -------------------------------------------------------------------
# Locate repository root regardless of notebook working directory.
# -------------------------------------------------------------------

def find_repo_root(
    start: Path | None = None,
    markers: tuple[str, ...] = (".git", "pyproject.toml", "setup.py"),
) -> Path:
    """
    Search upward from the working directory for a repository root.

    Parameters
    ----------
    start : Path | None
        Starting directory. Defaults to the current working directory.
    markers : tuple[str, ...]
        Files or directories that identify a project root.

    Returns
    -------
    Path
        Absolute path to the repository root.

    Raises
    ------
    FileNotFoundError
        If no repository root can be identified.
    """
    start = Path(start or Path.cwd()).resolve()

    if start.is_file():
        start = start.parent

    for directory in (start, *start.parents):
        if any((directory / marker).exists() for marker in markers):
            return directory

    raise FileNotFoundError(
        f"Could not locate repository root starting from {start}"
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

import pandas as pd

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\benca\code\PersonalPythonProjects\BigDataBowl2027\.venv\Scripts\python.exe
Repository root: C:\Users\benca\code\PersonalPythonProjects\BigDataBowl2027


In [2]:
combine_tracking_USER = pd.read_parquet(REPO_ROOT / "data" / "processed" / "combine_tracking_USER.parquet")

In [3]:
players = pd.read_csv(REPO_ROOT / "data" / "raw" / "nfl-big-data-bowl-2027" / "players.csv")

In [4]:
combine_tracking_USER_PLAYERS = combine_tracking_USER.merge(
    players.drop(columns=["draft_year"], errors="ignore"),
    on="nfl_id",
    how="left",
    validate="many_to_one"
)

In [5]:
import pandas as pd

# -------------------------------------------------------
# 1. Row count validation
# -------------------------------------------------------

n_before = len(combine_tracking_USER)
n_after = len(combine_tracking_USER_PLAYERS)

print("=== ROW COUNT VALIDATION ===")
print(f"Before merge: {n_before:,}")
print(f"After merge:  {n_after:,}")
print(f"Difference:   {n_after - n_before:+,}")

assert n_before == n_after, "FAIL: Merge changed row count!"
print("PASS: Row count preserved.\n")


# -------------------------------------------------------
# 2. Join key uniqueness
# -------------------------------------------------------

print("=== JOIN KEY VALIDATION ===")

join_keys = ["nfl_id", "draft_year"]

duplicates = players.duplicated(subset=join_keys).sum()

print(f"Duplicate player join keys: {duplicates:,}")

assert duplicates == 0, "FAIL: Duplicate player keys!"
print("PASS: Player join keys are unique.\n")


# -------------------------------------------------------
# 3. Player match coverage
# -------------------------------------------------------

print("=== PLAYER MATCH COVERAGE ===")

player_keys = players[join_keys].drop_duplicates()

coverage = combine_tracking_USER.merge(
    player_keys.assign(_matched=True),
    on=join_keys,
    how="left",
    validate="many_to_one"
)

# BALL observations may not represent independent players.
player_mask = combine_tracking_USER["entity_type"].eq("PLAYER")
eligible = coverage.loc[player_mask & coverage["nfl_id"].notna()]

matched = eligible["_matched"].fillna(False).astype(bool)

print(f"Eligible player observations: {len(eligible):,}")
print(f"Matched observations:         {matched.sum():,}")
print(f"Unmatched observations:       {(~matched).sum():,}")
print(f"Match rate:                   {matched.mean():.2%}" if len(matched) else "Match rate: N/A")

if (~matched).any():
    print("\nUnmatched player keys:")
    display(
        eligible.loc[~matched, join_keys]
        .drop_duplicates()
        .head(20)
    )


# -------------------------------------------------------
# 4. Missing values introduced by merge
# -------------------------------------------------------

print("\n=== PLAYER METADATA VALIDATION ===")

metadata_cols = [
    col for col in players.columns
    if col not in join_keys
    and col in combine_tracking_USER_PLAYERS.columns
]

if metadata_cols:
    missing_before = players[metadata_cols].isna().mean()
    missing_after = combine_tracking_USER_PLAYERS[metadata_cols].isna().mean()

    missing_summary = pd.DataFrame({
        "Source Missing %": missing_before * 100,
        "Merged Missing %": missing_after * 100,
    })

    display(missing_summary.round(2))
else:
    print("No additional player metadata columns found.")


# -------------------------------------------------------
# 5. Verify original tracking columns unchanged
# -------------------------------------------------------

print("=== TRACKING DATA INTEGRITY ===")

common_cols = combine_tracking_USER.columns.intersection(
    combine_tracking_USER_PLAYERS.columns
)

pd.testing.assert_frame_equal(
    combine_tracking_USER[common_cols].reset_index(drop=True),
    combine_tracking_USER_PLAYERS[common_cols].reset_index(drop=True),
    check_dtype=True,
)

print("PASS: Original tracking columns unchanged.")

=== ROW COUNT VALIDATION ===
Before merge: 463,189
After merge:  463,189
Difference:   +0
PASS: Row count preserved.

=== JOIN KEY VALIDATION ===
Duplicate player join keys: 0
PASS: Player join keys are unique.

=== PLAYER MATCH COVERAGE ===
Eligible player observations: 431,094
Matched observations:         431,094
Unmatched observations:       0
Match rate:                   100.00%

=== PLAYER METADATA VALIDATION ===


,Source Missing %,Merged Missing %
display_name,0.0,0.00
nfl_position,0.0,0.00
birth_date,0.0,0.00
college_name,0.0,0.00
college_conference,0.0,0.00
draft_round,24.9,25.91
draft_pick_within_round,24.9,25.91
draft_overall_pick,24.9,25.91


=== TRACKING DATA INTEGRITY ===
PASS: Original tracking columns unchanged.


In [6]:
combine_tracking_USER_PLAYERS_WRS = combine_tracking_USER_PLAYERS[combine_tracking_USER_PLAYERS["nfl_position"] == "WR"]
combine_tracking_USER_PLAYERS_WRS['drill_name'].unique()

array(['POST_CORNER_ROUTE_RIGHT', 'GAUNTLET_DRILL',
       'OVER_SHOULDER_ADJUST', 'GO_ROUTE_RIGHT', 'CIRCUS_ROUTE_LEFT',
       'SLANT_ROUTE_LEFT', 'SPEED_OUT_ROUTE_LEFT', 'CURL_ROUTE_RIGHT',
       'DAGGER_ROUTE_LEFT', 'SLOT_STRIKE_ROUTE_LEFT',
       'COMEBACK_ROUTE_RIGHT', 'SLOT_SAIL_ROUTE_RIGHT',
       'RED_ZONE_FADE_RIGHT', 'FORTY_YARD_DASH', 'THREE_CONE_DRILL',
       'SHORT_SHUTTLE', 'SPEED_OUT_LEFT', 'POST_CORNER_ROUTE_LEFT'],
      dtype=object)

In [8]:
import importlib
import src.drill_animation as drill_animation

importlib.reload(drill_animation)

from IPython.display import HTML, display

config = drill_animation.AnimationConfig(
    fps=1,
    show_ball=True,
    trail_alpha=0.07,

    # New histogram configuration
    histogram_column="USER_curvature",
    histogram_bins=25,
)

animation, drill_processed = drill_animation.animate_drill(
    tracking_df=combine_tracking_USER_PLAYERS_WRS,
    drill_name="POST_CORNER_ROUTE_RIGHT",
    config=config,
)

display(HTML(animation.to_jshtml(fps=config.fps)))